In [1]:
import pandas as pd
import numpy as np

In [2]:
ICFES = pd.read_csv(r"C:\Users\DanielGP\OneDrive - Caja de Compensacion Familiar de Antioquia COMFAMA\Documentos\Resultados_únicos_Saber_11_20251205.csv")

ICFES.info()

C:\Users\DanielGP\AppData\Local\Temp\ipykernel_41480\2671024660.py:1: DtypeWarning: Columns (45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  ICFES = pd.read_csv(r"C:\Users\DanielGP\OneDrive - Caja de Compensacion Familiar de Antioquia COMFAMA\Documentos\Resultados_únicos_Saber_11_20251205.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7109704 entries, 0 to 7109703
Data columns (total 51 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   PERIODO                        int64  
 1   ESTU_TIPODOCUMENTO             object 
 2   ESTU_CONSECUTIVO               object 
 3   COLE_AREA_UBICACION            object 
 4   COLE_BILINGUE                  object 
 5   COLE_CALENDARIO                object 
 6   COLE_CARACTER                  object 
 7   COLE_COD_DANE_ESTABLECIMIENTO  float64
 8   COLE_COD_DANE_SEDE             float64
 9   COLE_COD_DEPTO_UBICACION       float64
 10  COLE_COD_MCPIO_UBICACION       float64
 11  COLE_CODIGO_ICFES              float64
 12  COLE_DEPTO_UBICACION           object 
 13  COLE_GENERO                    object 
 14  COLE_JORNADA                   object 
 15  COLE_MCPIO_UBICACION           object 
 16  COLE_NATURALEZA                object 
 17  COLE_NOMBRE_ESTABLECIMIENTO    object 
 18  CO

In [13]:
ICFES.head(100
           )

,PERIODO,ESTU_TIPODOCUMENTO,ESTU_CONSECUTIVO,COLE_AREA_UBICACION,COLE_BILINGUE,COLE_CALENDARIO,COLE_CARACTER,COLE_COD_DANE_ESTABLECIMIENTO,COLE_COD_DANE_SEDE,COLE_COD_DEPTO_UBICACION,...,FAMI_TIENEINTERNET,FAMI_TIENELAVADORA,DESEMP_INGLES,PUNT_INGLES,PUNT_MATEMATICAS,PUNT_SOCIALES_CIUDADANAS,PUNT_C_NATURALES,PUNT_LECTURA_CRITICA,PUNT_GLOBAL,PERIODO_NUM
0,20131,CR,SB11201310000414,URBANO,N,B,ACADÉMICO,3.118480e+11,3.118480e+11,11.0,...,Si,Si,B+,94.0,88.0,NaN,NaN,NaN,NaN,20131
1,20194,TI,SB11201940464873,RURAL,N,A,TÉCNICO/ACADÉMICO,1.410160e+11,2.410160e+11,41.0,...,Si,Si,B1,71.0,66.0,70.0,65.0,69.0,339.0,20194
2,20194,TI,SB11201940464873,RURAL,N,A,TÉCNICO/ACADÉMICO,1.410160e+11,2.410160e+11,41.0,...,Si,Si,B1,71.0,66.0,70.0,65.0,69.0,339.0,20194
3,20122,TI,SB11201220204399,URBANO,N,A,TÉCNICO/ACADÉMICO,1.631300e+11,1.631300e+11,63.0,...,Si,No,A1,48.0,45.0,NaN,NaN,NaN,NaN,20122
4,20132,TI,SB11201320464198,URBANO,N,A,TÉCNICO,1.190010e+11,1.190010e+11,19.0,...,Si,Si,A-,43.0,52.0,NaN,NaN,NaN,NaN,20132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,20122,TI,SB11201220350729,URBANO,N,A,TÉCNICO/ACADÉMICO,1.110010e+11,1.110010e+11,11.0,...,Si,Si,A-,43.0,50.0,NaN,NaN,NaN,NaN,20122
96,20142,TI,SB11201420498565,RURAL,N,A,ACADÉMICO,2.231890e+11,2.231890e+11,23.0,...,No,No,A-,43.0,48.0,42.0,51.0,44.0,230.0,20142
97,20122,TI,SB11201220007144,URBANO,N,A,ACADÉMICO,3.680010e+11,3.680010e+11,68.0,...,Si,Si,A1,51.0,61.0,NaN,NaN,NaN,NaN,20122
98,20112,TI,SB11201120149128,URBANO,N,A,TÉCNICO/ACADÉMICO,1.053600e+11,1.053600e+11,5.0,...,Si,Si,A2,58.0,45.0,NaN,NaN,NaN,NaN,20112


In [19]:
import pandas as pd

# 1. Filtrar por Bogotá y PERIODO > 2020
mask_bog = ICFES['COLE_MCPIO_UBICACION'].astype(str).str.lower().str.contains('bogot', na=False)
mask_periodo = pd.to_numeric(ICFES['PERIODO'], errors='coerce') > 2020

df_bog = ICFES[mask_bog & mask_periodo].copy()
print(f"Registros filtrados: {len(df_bog)}")

# 2. Normalizar COLE_NATURALEZA y unificar categorías
df_bog['COLE_NATURALEZA'] = (
    df_bog['COLE_NATURALEZA']
      .astype(str)
      .str.upper()
      .str.strip()
      .replace({
          'NOOFICIAL': 'NO OFICIAL',
          'NO_OFICIAL': 'NO OFICIAL'
      })
)

# Quedarnos solo con OFICIAL y NO OFICIAL
df_bog = df_bog[df_bog['COLE_NATURALEZA'].isin(['OFICIAL', 'NO OFICIAL'])]

# 3. Opcional: quitar estrato nulo
col = 'FAMI_ESTRATOVIVIENDA'
df_bog = df_bog[df_bog[col].notna()]

# 4. Calcular conteos y porcentajes por naturaleza y estrato
counts = (
    df_bog
    .groupby(['COLE_NATURALEZA', col])
    .size()
    .reset_index(name='count')
)

# Porcentaje dentro de cada naturaleza
counts['percent'] = (
    counts['count'] /
    counts.groupby('COLE_NATURALEZA')['count'].transform('sum') * 100
)

# Ordenar para ver mejor
counts = counts.sort_values(['COLE_NATURALEZA', 'percent'], ascending=[True, False])
display(counts)

# 5. Tabla pivote: filas = estrato, columnas = naturaleza, valores = %
pivot = (
    df_bog
    .pivot_table(index=col, columns='COLE_NATURALEZA', aggfunc='size', fill_value=0)
)

pivot = (pivot / pivot.sum(axis=0) * 100).round(2)

print("\nPorcentaje por FAMI_ESTRATOVIVIENDA dentro de cada naturaleza:")
display(pivot)

Registros filtrados: 1131725


,COLE_NATURALEZA,FAMI_ESTRATOVIVIENDA,count,percent
2,NO OFICIAL,Estrato 3,218209,41.166538
1,NO OFICIAL,Estrato 2,172082,32.464382
3,NO OFICIAL,Estrato 4,73969,13.954730
4,NO OFICIAL,Estrato 5,25321,4.776970
0,NO OFICIAL,Estrato 1,21699,4.093657
5,NO OFICIAL,Estrato 6,17584,3.317335
6,NO OFICIAL,Sin Estrato,1200,0.226388
8,OFICIAL,Estrato 2,311532,54.158286
9,OFICIAL,Estrato 3,156100,27.137207
7,OFICIAL,Estrato 1,96880,16.842105



Porcentaje por FAMI_ESTRATOVIVIENDA dentro de cada naturaleza:


COLE_NATURALEZA,NO OFICIAL,OFICIAL
FAMI_ESTRATOVIVIENDA,,
Estrato 1,4.09,16.84
Estrato 2,32.46,54.16
Estrato 3,41.17,27.14
Estrato 4,13.95,1.27
Estrato 5,4.78,0.20
Estrato 6,3.32,0.10
Sin Estrato,0.23,0.29


In [21]:
# Aseguramos PERIODO_NUM
ICFES['PERIODO_NUM'] = pd.to_numeric(ICFES['PERIODO'], errors='coerce')

# Filtramos solo Bogotá 2021, sin tocar naturaleza todavía
mask_bog = ICFES['COLE_MCPIO_UBICACION'].astype(str).str.lower().str.contains('bogot', na=False)
mask_2021 = (ICFES['PERIODO_NUM'] // 10) == 2021

df_bog_2021 = ICFES[mask_bog & mask_2021].copy()

# Normalizamos texto de naturaleza solo para ver qué hay
nat_raw = (
    df_bog_2021['COLE_NATURALEZA']
    .astype(str)
    .str.upper()
    .str.strip()
)

print("TOP 20 COLE_NATURALEZA en Bogotá 2021:")
print(nat_raw.value_counts().head(20))

TOP 20 COLE_NATURALEZA en Bogotá 2021:
COLE_NATURALEZA
NO OFICIAL    3245
OFICIAL         14
Name: count, dtype: int64


In [23]:
import pandas as pd

# 1. Asegurar PERIODO_NUM
ICFES['PERIODO_NUM'] = pd.to_numeric(ICFES['PERIODO'], errors='coerce')

# 2. Filtrar Bogotá y PERIODO del año 2021 sin agruparlos
mask_bog = ICFES['COLE_MCPIO_UBICACION'].astype(str).str.lower().str.contains('bogot', na=False)

# PERIODO tipo 20211, 20212, ..., 20214
mask_2021 = (ICFES['PERIODO_NUM'] >= 20211) & (ICFES['PERIODO_NUM'] <= 20215)

df_bog_2021_raw = ICFES[mask_bog & mask_2021].copy()

print("Total registros Bogotá 2021 (sin limpiar naturaleza):", len(df_bog_2021_raw))

# 3. Ver distribución de COLE_NATURALEZA exacta para 2021
print("\nTop COLE_NATURALEZA exacto para Bogotá 2021:")
print(df_bog_2021_raw['COLE_NATURALEZA'].value_counts().head(20))

# 4. Conteo por PERIODO exacto (20211, 20212…)
print("\nConteo por PERIODO (20211, 20212, etc):")
print(df_bog_2021_raw['PERIODO'].value_counts().sort_index())

# 5. Conteo solo de OFICIAL y NO OFICIAL tal como aparecen en el archivo
mask_oficial = df_bog_2021_raw['COLE_NATURALEZA'] == "OFICIAL"
mask_nooficial = df_bog_2021_raw['COLE_NATURALEZA'] == "NO OFICIAL"

print("\nRegistros OFICIAL (exacto):", mask_oficial.sum())
print("Registros NO OFICIAL (exacto):", mask_nooficial.sum())

# 6. Tabla estratos SOLO con datos exactos y sin imputación
df_bog_2021_raw['FAMI_ESTRATOVIVIENDA'] = df_bog_2021_raw['FAMI_ESTRATOVIVIENDA'].astype(str)

counts_exact = (
    df_bog_2021_raw[df_bog_2021_raw['COLE_NATURALEZA'].isin(["OFICIAL", "NO OFICIAL"])]
    .groupby(['COLE_NATURALEZA', 'FAMI_ESTRATOVIVIENDA'])
    .size()
    .reset_index(name='count')
)

print("\nTabla de estratos para OFICIAL y NO OFICIAL (datos exactos, sin limpiar):")
display(counts_exact)

Total registros Bogotá 2021 (sin limpiar naturaleza): 3259

Top COLE_NATURALEZA exacto para Bogotá 2021:
COLE_NATURALEZA
NO OFICIAL    3245
OFICIAL         14
Name: count, dtype: int64

Conteo por PERIODO (20211, 20212, etc):
PERIODO
20211    3259
Name: count, dtype: int64

Registros OFICIAL (exacto): 14
Registros NO OFICIAL (exacto): 3245

Tabla de estratos para OFICIAL y NO OFICIAL (datos exactos, sin limpiar):


,COLE_NATURALEZA,FAMI_ESTRATOVIVIENDA,count
0,NO OFICIAL,Estrato 1,41
1,NO OFICIAL,Estrato 2,213
2,NO OFICIAL,Estrato 3,352
3,NO OFICIAL,Estrato 4,736
4,NO OFICIAL,Estrato 5,824
5,NO OFICIAL,Estrato 6,959
6,NO OFICIAL,Sin Estrato,40
7,NO OFICIAL,nan,80
8,OFICIAL,Estrato 1,1
9,OFICIAL,Estrato 2,6


In [20]:
import pandas as pd

# 1. Aseguramos PERIODO numérico
ICFES['PERIODO_NUM'] = pd.to_numeric(ICFES['PERIODO'], errors='coerce')

# 2. Filtrar por Bogotá y PERIODO > 2020
mask_bog = ICFES['COLE_MCPIO_UBICACION'].astype(str).str.lower().str.contains('bogot', na=False)
mask_periodo = ICFES['PERIODO_NUM'] > 2010

df_bog = ICFES[mask_bog & mask_periodo].copy()
print(f"Registros filtrados: {len(df_bog)}")

# 3. Crear columna AÑO a partir de PERIODO (ej: 20201 -> 2020)
df_bog['AÑO'] = (df_bog['PERIODO_NUM'] // 10).astype(int)

# 4. Normalizar COLE_NATURALEZA
df_bog['COLE_NATURALEZA'] = (
    df_bog['COLE_NATURALEZA'].astype(str).str.upper().str.strip()
      .replace({'NOOFICIAL': 'NO OFICIAL', 'NO_OFICIAL': 'NO OFICIAL'})
)

# Quedarnos solo con OFICIAL y NO OFICIAL
df_bog = df_bog[df_bog['COLE_NATURALEZA'].isin(['OFICIAL', 'NO OFICIAL'])]

# 5. Quitar estrato nulo
col = 'FAMI_ESTRATOVIVIENDA'
df_bog = df_bog[df_bog[col].notna()]

# =============================
# A) CONTEXTO GENERAL (SIN AÑO)
# =============================

counts = (
    df_bog.groupby(['COLE_NATURALEZA', col])
           .size()
           .reset_index(name='count')
)

counts['percent'] = (
    counts['count'] /
    counts.groupby('COLE_NATURALEZA')['count'].transform('sum') * 100
)

print("\nDistribución general por naturaleza:")
display(counts)


# =============================
# B) AGRUPACIÓN POR AÑO
# =============================

# Conteo y porcentaje por AÑO + NATURALEZA + ESTRATO
counts_year = (
    df_bog.groupby(['AÑO', 'COLE_NATURALEZA', col])
           .size()
           .reset_index(name='count')
)

counts_year['percent'] = (
    counts_year['count'] /
    counts_year.groupby(['AÑO', 'COLE_NATURALEZA'])['count'].transform('sum') * 100
)

counts_year = counts_year.sort_values(['AÑO', 'COLE_NATURALEZA', 'percent'], ascending=[True, True, False])

print("\nPorcentajes por año, naturaleza y estrato:")
display(counts_year)   # <<---- AQUÍ YA SALE EL AÑO CORRECTAMENTE

# =============================
# C) TABLA TIPO MATRIZ (PIVOT)
# =============================

pivot_year = counts_year.pivot_table(
    index=['AÑO', col],
    columns='COLE_NATURALEZA',
    values='percent'
).round(2)

print("\nTabla pivot por año (con AÑO incluido):")
display(pivot_year)    # <<---- AQUÍ TAMBIÉN SALE EL AÑO

Registros filtrados: 1131725

Distribución general por naturaleza:

Distribución general por naturaleza:


,COLE_NATURALEZA,FAMI_ESTRATOVIVIENDA,count,percent
0,NO OFICIAL,Estrato 1,21699,4.093657
1,NO OFICIAL,Estrato 2,172082,32.464382
2,NO OFICIAL,Estrato 3,218209,41.166538
3,NO OFICIAL,Estrato 4,73969,13.954730
4,NO OFICIAL,Estrato 5,25321,4.776970
5,NO OFICIAL,Estrato 6,17584,3.317335
6,NO OFICIAL,Sin Estrato,1200,0.226388
7,OFICIAL,Estrato 1,96880,16.842105
8,OFICIAL,Estrato 2,311532,54.158286
9,OFICIAL,Estrato 3,156100,27.137207



Porcentajes por año, naturaleza y estrato:


,AÑO,COLE_NATURALEZA,FAMI_ESTRATOVIVIENDA,count,percent
2,2010,NO OFICIAL,Estrato 3,20158,43.328175
1,2010,NO OFICIAL,Estrato 2,15236,32.748689
3,2010,NO OFICIAL,Estrato 4,6085,13.079271
0,2010,NO OFICIAL,Estrato 1,2071,4.451466
4,2010,NO OFICIAL,Estrato 5,1899,4.081764
...,...,...,...,...,...
148,2022,OFICIAL,Estrato 1,12383,14.767334
151,2022,OFICIAL,Estrato 4,2019,2.407756
154,2022,OFICIAL,Sin Estrato,732,0.872946
152,2022,OFICIAL,Estrato 5,366,0.436473



Tabla pivot por año (con AÑO incluido):


COLE_NATURALEZA            NO OFICIAL  OFICIAL
AÑO  FAMI_ESTRATOVIVIENDA                     
2010 Estrato 1                   4.45    18.55
     Estrato 2                  32.75    53.33
     Estrato 3                  43.33    27.04
     Estrato 4                  13.08     0.85
     Estrato 5                   4.08     0.14
...                               ...      ...
2022 Estrato 3                  43.60    29.10
     Estrato 4                  15.18     2.41
     Estrato 5                   4.18     0.44
     Estrato 6                   2.53     0.20
     Sin Estrato                 0.65     0.87

[84 rows x 2 columns]

In [16]:
# Aseguramos PERIODO_NUM
ICFES['PERIODO_NUM'] = pd.to_numeric(ICFES['PERIODO'], errors='coerce')

# Filtro solo por ciudad, sin periodo todavía
mask_cart_ciudad = ICFES['COLE_MCPIO_UBICACION'].astype(str).str.lower().str.contains('cartagena', na=False)
print("Registros con 'cartagena' en COLE_MCPIO_UBICACION:", mask_cart_ciudad.sum())

print("\nEjemplos de COLE_MCPIO_UBICACION que contienen 'cartagena':")
print(ICFES.loc[mask_cart_ciudad, 'COLE_MCPIO_UBICACION'].dropna().unique()[:20])

# Ahora con PERIODO > 2020
mask_periodo = ICFES['PERIODO_NUM'] > 2015
mask_cart_total = mask_cart_ciudad & mask_periodo
print("\nRegistros de Cartagena con PERIODO > 2020:", mask_cart_total.sum())

Registros con 'cartagena' en COLE_MCPIO_UBICACION: 167408

Ejemplos de COLE_MCPIO_UBICACION que contienen 'cartagena':
['CARTAGENA' 'CARTAGENA DE INDIAS' 'CARTAGENA DEL CHAIRA'
 'CARTAGENA DEL CHAIRÁ']

Registros de Cartagena con PERIODO > 2020: 167408


In [17]:
import pandas as pd

# Asegurar PERIODO_NUM
ICFES['PERIODO_NUM'] = pd.to_numeric(ICFES['PERIODO'], errors='coerce')

# --- 1. Filtrar SOLO Cartagena real (no Caquetá/Caquetá Chairá) ---

# Normalizamos municipio
ICFES['MUNICIPIO_NORM'] = (
    ICFES['COLE_MCPIO_UBICACION']
    .astype(str)
    .str.upper()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.strip()
)

# Quedar solo con estos dos
validos = ['CARTAGENA', 'CARTAGENA DE INDIAS']

df_cart = ICFES[
    (ICFES['MUNICIPIO_NORM'].isin(validos)) &
    (ICFES['PERIODO_NUM'] > 2015)
].copy()

print("Registros válidos solo de Cartagena (ya excluye Chairá):", len(df_cart))

# --- 2. Unificar ambos nombres a CARTAGENA ---
df_cart['MUNICIPIO_NORM'] = 'CARTAGENA'

# --- 3. Extraer año ---
df_cart['AÑO'] = (df_cart['PERIODO_NUM'] // 10).astype(int)

# --- 4. Normalizar naturaleza ---
df_cart['COLE_NATURALEZA'] = (
    df_cart['COLE_NATURALEZA'].astype(str).str.upper().str.strip()
    .replace({'NOOFICIAL': 'NO OFICIAL', 'NO_OFICIAL': 'NO OFICIAL'})
)

df_cart = df_cart[df_cart['COLE_NATURALEZA'].isin(['OFICIAL', 'NO OFICIAL'])]

# --- 5. Quitar estrato nulo ---
col_estrato = 'FAMI_ESTRATOVIVIENDA'
df_cart = df_cart[df_cart[col_estrato].notna()]

print("Registros después de filtrar naturaleza y estrato:", len(df_cart))

# =============================
# A) Distribución general
# =============================
counts_cart = (
    df_cart
    .groupby(['COLE_NATURALEZA', col_estrato])
    .size()
    .reset_index(name='count')
)

counts_cart['percent'] = (
    counts_cart['count'] /
    counts_cart.groupby('COLE_NATURALEZA')['count'].transform('sum') * 100
)

print("\nDistribución general por naturaleza (Cartagena):")
display(counts_cart)

# =============================
# B) Distribución por año
# =============================
counts_year_cart = (
    df_cart
    .groupby(['AÑO', 'COLE_NATURALEZA', col_estrato])
    .size()
    .reset_index(name='count')
)

counts_year_cart['percent'] = (
    counts_year_cart['count'] /
    counts_year_cart.groupby(['AÑO', 'COLE_NATURALEZA'])['count'].transform('sum') * 100
)

print("\nDistribución por año, naturaleza y estrato (Cartagena):")
display(counts_year_cart)

# =============================
# C) Tabla pivot
# =============================
pivot_year_cart = counts_year_cart.pivot_table(
    index=['AÑO', col_estrato],
    columns='COLE_NATURALEZA',
    values='percent'
).round(2)

print("\nTabla pivot por año (Cartagena):")
display(pivot_year_cart)

Registros válidos solo de Cartagena (ya excluye Chairá): 164879
Registros después de filtrar naturaleza y estrato: 157249

Distribución general por naturaleza (Cartagena):
Registros después de filtrar naturaleza y estrato: 157249

Distribución general por naturaleza (Cartagena):


,COLE_NATURALEZA,FAMI_ESTRATOVIVIENDA,count,percent
0,NO OFICIAL,Estrato 1,11352,22.242251
1,NO OFICIAL,Estrato 2,15793,30.943611
2,NO OFICIAL,Estrato 3,13424,26.301971
3,NO OFICIAL,Estrato 4,5385,10.550962
4,NO OFICIAL,Estrato 5,2784,5.454759
5,NO OFICIAL,Estrato 6,1957,3.834398
6,NO OFICIAL,Sin Estrato,343,0.672048
7,OFICIAL,Estrato 1,58988,55.538504
8,OFICIAL,Estrato 2,31485,29.643822
9,OFICIAL,Estrato 3,10890,10.253175



Distribución por año, naturaleza y estrato (Cartagena):


,AÑO,COLE_NATURALEZA,FAMI_ESTRATOVIVIENDA,count,percent
0,2010,NO OFICIAL,Estrato 1,1173,26.695494
1,2010,NO OFICIAL,Estrato 2,1369,31.156122
2,2010,NO OFICIAL,Estrato 3,1066,24.260355
3,2010,NO OFICIAL,Estrato 4,420,9.558489
4,2010,NO OFICIAL,Estrato 5,215,4.893036
...,...,...,...,...,...
149,2022,OFICIAL,Estrato 3,2564,14.608854
150,2022,OFICIAL,Estrato 4,724,4.125121
151,2022,OFICIAL,Estrato 5,222,1.264885
152,2022,OFICIAL,Estrato 6,152,0.866048



Tabla pivot por año (Cartagena):


COLE_NATURALEZA            NO OFICIAL  OFICIAL
AÑO  FAMI_ESTRATOVIVIENDA                     
2010 Estrato 1                  26.70    64.41
     Estrato 2                  31.16    27.05
     Estrato 3                  24.26     7.78
     Estrato 4                   9.56     0.60
     Estrato 5                   4.89     0.06
...                               ...      ...
2022 Estrato 3                  28.91    14.61
     Estrato 4                  10.34     4.13
     Estrato 5                   3.80     1.26
     Estrato 6                   2.21     0.87
     Sin Estrato                 1.98     5.23

[84 rows x 2 columns]